# 📊 Trabalho de Clusterização em 3D com KMeans

## Disciplina: Aprendizado de Máquina Não Supervisionado  
**Curso:** Tecnologia em Ciência de Dados  
**Instituição:** Faculdade de Tecnologia e Inovação Senac DF  
**Professor:** Rogério Gomes Lopes  

**Objetivo:** Analisar a **Distribuição de Renda por Centis no Brasil** a partir de dados **granulares por Estado, Região e Ano**,  
por meio de **clusterização não supervisionada (KMeans)**, utilizando **3 variáveis contínuas** e visualização em **3D**.  
O propósito é identificar **padrões de renda, patrimônio e carga tributária** entre diferentes grupos socioeconômicos,  
explorando como esses perfis se distribuem entre estados e regiões, bem como sua evolução temporal.

---

### 👨‍🎓 Alunos
- Anderson de Matos Guimarães  
- Gustavo Stefano Thomazinho  
- Renan Ost  

---

📅 **Semestre:** 4º semestre (2025/2)  


## 1. Business Understanding

O presente trabalho busca analisar a **Distribuição de Renda por Centis no Brasil**, utilizando técnicas de 
**Aprendizado de Máquina Não Supervisionado**, com foco na **clusterização em 3D via KMeans**.

A base de dados disponibilizada pela Receita Federal apresenta informações **granulares** de renda declarada, 
discriminadas por:

- **Ano-calendário** (período de referência da declaração);
- **Ente Federativo (UF)**;
- **Centil de Renda** (100 divisões iguais da população, sendo que o centésimo é subdividido em extratos mais finos).

---

### 🔎 O que é um Centil?

Na base da Receita Federal, os declarantes são divididos em **100 grupos de igual tamanho**, chamados **centis**, 
a partir da **Renda Tributável Bruta (RTB)**:

- O **1º centil** corresponde ao 1% da população com menor renda declarada.  
- O **100º centil** corresponde ao 1% mais rico, que é subdividido em 10 partes iguais; e a última dessas (o 0,1% do topo) 
é novamente dividida em 10 partes, para discriminar com mais detalhe o **extrato superior da renda**.

Essa metodologia, baseada no conceito estatístico de **percentis**, permite analisar a distribuição de renda de forma 
**granular e comparável**, garantindo que cada centil represente a mesma quantidade de contribuintes.  
A subdivisão do centésimo centil é uma decisão metodológica da Receita Federal para melhor captar a 
**alta concentração de renda no topo**.

---

### Variáveis disponíveis na base
Para cada (Ano × UF × Centil), são informados valores como:

- **Rendimentos Tributáveis** (soma, limite, acumulada);  
- **Rendimentos Sujeitos à Tributação Exclusiva**;  
- **Rendimentos Isentos** (dividendos, Simples, outros);  
- **Despesas Dedutíveis** (saúde, educação, previdência, pensão, etc.);  
- **Imposto Devido**;  
- **Bens e Direitos** (imóveis, móveis, financeiros, outros);  
- **Dívidas e Ônus**.  

---

### 🎯 Objetivo da Análise
O objetivo é **identificar padrões de desigualdade econômica no Brasil**, por meio da clusterização de observações 
granulares (Centil × UF × Ano), utilizando **3 variáveis contínuas derivadas**:

1. **Renda Total** = Rendimentos Tributáveis + Exclusivos + Isentos  
2. **Patrimônio Líquido** = Bens Totais – Dívidas e Ônus  
3. **Carga Tributária Efetiva** = Imposto Devido / Renda Total  

---

### Questões de análise
- Quais perfis socioeconômicos são revelados pelos clusters?  
- Como esses clusters se distribuem entre **estados** e **regiões**?  
- Há diferenças significativas quando comparamos **anos distintos** (análise temporal)?  
- Os clusters ajudam a revelar **padrões de desigualdade de renda, patrimônio e carga tributária** no Brasil?


## 2. Data Understanding

Nesta etapa buscamos compreender a estrutura do dataset original, sua granularidade e o significado de suas variáveis.  
O objetivo é garantir que conhecemos bem os dados antes de realizar qualquer preparação ou modelagem.

---

### 📂 Dataset utilizado
A base **Distribuição de Renda por Centis** é disponibilizada pela Receita Federal.  
Cada linha corresponde a uma combinação de:

- **Ano-calendário** (período da declaração);  
- **Ente Federativo (UF)**;  
- **Centil de Renda** (100 divisões iguais da população com base na Renda Tributável Bruta).  

Ou seja, temos dados **granulares** por **ano, estado e centil**, o que permite uma análise detalhada da distribuição de renda no Brasil.

---

### 📖 Principais variáveis (segundo o dicionário de dados)
Algumas variáveis relevantes, extraídas do dicionário da Receita Federal:contentReference[oaicite:0]{index=0}, são:

- **Ano-calendário:** ano a que se refere a declaração.  
- **Ente Federativo:** estado (UF) ao qual os dados se referem.  
- **Centil:** grupo que representa 1% dos declarantes ordenados pela Renda Tributável Bruta.  
- **Rendimentos Tributáveis – Soma da RTB (R$ mi):** total da renda tributável bruta dentro do centil.  
- **Rendimentos Sujeitos à Tributação Exclusiva (R$ mi):** rendimentos sujeitos à tributação exclusiva e definitiva.  
- **Rendimentos Isentos (R$ mi):** incluem dividendos, Simples e outros rendimentos isentos.  
- **Despesas Dedutíveis (R$ mi):** previdência, dependentes, instrução, médicas, pensão, livro-caixa.  
- **Imposto Devido (R$ mi):** total de imposto calculado para o centil.  
- **Bens e Direitos (R$ mi):** imóveis, móveis, financeiros, outros.  
- **Dívidas e Ônus (R$ mi):** valor total de dívidas declaradas.  

Essas variáveis servirão de base para criarmos as **três variáveis derivadas** que serão utilizadas na clusterização.

---

### 🔎 Exploração inicial do dataset
Antes de criar variáveis derivadas, vamos observar o dataset original.


In [ ]:
# ================================
# Carregar o dataset original
# ================================
import pandas as pd

# Como o arquivo está no mesmo diretório do notebook:
csv_path = "distribuicao-renda.csv"

# Em alguns casos o separador pode ser ";", por isso testamos
df = pd.read_csv(csv_path, sep=";", low_memory=False)


In [5]:
print("Dimensões do dataset:", df.shape)


Dimensões do dataset: (46350, 24)


In [6]:
# Primeiras linhas do dataset
df.head()


,Ano-calendário,Ente Federativo,Centil,Quantidade de Contribuintes,Rendimentos Tributaveis - Limite Superior da RTB do Centil [R$ milhões],Rendimentos Tributaveis - Soma da RTB do Centil [R$ milhões],Rendimentos Tributaveis - RTB Acumulada do Centil [R$ milhões],Rendimentos Tributaveis - Média da RTB do Centil [R$],Rendimentos Sujeitos à Tribut. Exclusiva [R$ milhões],Rendimentos Isentos - Lucros e dividendos [R$ milhões],...,Despesas Dedutíveis - Instrução [R$ milhões],Despesas Dedutíveis - Médicas [R$ milhões],Despesas Dedutíveis - Pensão Alimentícia [R$ milhões],Despesas Dedutíveis - Livro-Caixa [R$ milhões],Imposto Devido [R$ milhões],Bens e Direitos - Imóveis [R$ milhões],Bens e Direitos - Móveis [R$ milhões],Bens e Direitos - Financeiros [R$ milhões],Bens e Direitos - Outros Bens e Direitos [R$ milhões],Dívidas e Ônus [R$ milhões]
0,2006,BRASIL,1,241.563,NaN,NaN,NaN,NaN,"235,61","481,27",...,NaN,NaN,NaN,NaN,"0,16","5.281,59","686,21","6.549,15","1.006,40","1.610,39"
1,2006,BRASIL,2,241.563,NaN,NaN,NaN,NaN,"208,74","483,44",...,NaN,NaN,NaN,NaN,"0,22","5.295,48","668,82","5.762,77","681,75","694,12"
2,2006,BRASIL,3,241.562,NaN,NaN,NaN,NaN,"219,96","459,87",...,NaN,NaN,NaN,NaN,"0,31","5.566,27","670,64","5.451,95","377,17","650,98"
3,2006,BRASIL,4,241.563,NaN,NaN,NaN,NaN,"257,01","481,93",...,NaN,NaN,NaN,NaN,"0,17","5.860,02","678,44","6.104,09","256,16","1.079,20"
4,2006,BRASIL,5,241.562,NaN,NaN,NaN,NaN,"249,88","464,23",...,NaN,NaN,NaN,NaN,"0,17","5.193,31","682,38","5.592,52","269,28","671,97"


📌 **Análise do head():**  
- Confirma a granularidade dos dados: cada linha corresponde a **Ano × Ente Federativo × Centil**.  
- Observa-se que o campo **Ente Federativo = BRASIL** representa o nível agregado nacional, que deverá ser removido
nas análises, pois o foco é a comparação por estados (UF).  
- Colunas principais visíveis: rendimentos tributáveis, exclusivos, isentos, despesas dedutíveis, imposto devido, bens e direitos, dívidas.  
- Valores em **milhões de R$** (atenção na interpretação dos números).  
- Alguns campos aparecem como `NaN` no agregado “Brasil”, reforçando a necessidade de filtrar apenas os dados de UF.  


In [7]:
# Últimas linhas do dataset
df.tail()


,Ano-calendário,Ente Federativo,Centil,Quantidade de Contribuintes,Rendimentos Tributaveis - Limite Superior da RTB do Centil [R$ milhões],Rendimentos Tributaveis - Soma da RTB do Centil [R$ milhões],Rendimentos Tributaveis - RTB Acumulada do Centil [R$ milhões],Rendimentos Tributaveis - Média da RTB do Centil [R$],Rendimentos Sujeitos à Tribut. Exclusiva [R$ milhões],Rendimentos Isentos - Lucros e dividendos [R$ milhões],...,Despesas Dedutíveis - Instrução [R$ milhões],Despesas Dedutíveis - Médicas [R$ milhões],Despesas Dedutíveis - Pensão Alimentícia [R$ milhões],Despesas Dedutíveis - Livro-Caixa [R$ milhões],Imposto Devido [R$ milhões],Bens e Direitos - Imóveis [R$ milhões],Bens e Direitos - Móveis [R$ milhões],Bens e Direitos - Financeiros [R$ milhões],Bens e Direitos - Outros Bens e Direitos [R$ milhões],Dívidas e Ônus [R$ milhões]
46345,2020,TO,100.6,169.0,"486.602,57","79,96","434,49","473.152,32","7,78","2,64",...,"0,44","3,2","0,94","2,29","16,12","185,25","16,75","61,71","13,29","27,73"
46346,2020,TO,100.7,169.0,"523.830,78","85,51",520,"505.995,90","8,91","3,38",...,"0,43","3,4","1,21","2,46","17,67","195,46","22,77","80,08","3,55","43,45"
46347,2020,TO,100.8,169.0,"589.084,62","93,48","613,49","553.159,76","11,9","12,9",...,"0,52","3,11","1,01","3,35","19,68","457,81","28,19","169,15","25,91","43,73"
46348,2020,TO,100.9,169.0,"749.691,30","110,74","724,23","655.252,33","6,82","12,17",...,"0,39","2,25","1,58","8,62","23,19","182,83","28,11","122,92","26,32","26,01"
46349,2020,TO,100.10,168.0,"18.277.661,84","238,8","963,03","1.421.432,54","904,35","16,15",...,"0,28","2,21","1,06","58,95","45,22","514,07","31,17","1.137,50","219,8","190,09"


📌 **Análise do tail():**  
- Confirma que o dataset cobre o período até **2020**.  
- Mostra registros do estado de **Tocantins (TO)** no **100º centil** subdividido (100.1 até 100.10).  
- Essa subdivisão é feita apenas no topo da distribuição para detalhar melhor a concentração de renda no 1% mais rico.  
- Diferentemente do `head()`, aqui não aparecem linhas agregadas como "BRASIL".  
- Os valores estão preenchidos, mas com magnitudes muito elevadas (na casa de milhões), evidenciando a concentração de renda no topo.  
- Não há linhas adicionais de “Total” ou agregados no final, apenas dados regulares.  


In [10]:
# ================================
# Estatísticas descritivas
# ================================
df.describe().T


,count,mean,std,min,25%,50%,75%,max
Ano-calendário,46350.0,2013.000000,4.320540,2006.000,2009.000,2013.00,2017.000,2020.0
Quantidade de Contribuintes,46350.0,96.555265,197.999486,1.001,2.894,6.79,29.169,993.0


📌 **Análise do describe():**  
- O resumo estatístico (`describe()`) apresenta apenas duas variáveis: **Ano-calendário** e **Quantidade de Contribuintes**.  
- Isso acontece porque as demais colunas (valores de rendimentos, bens, dívidas etc.) foram carregadas como **texto (string)**,  
  já que utilizam **vírgula como separador decimal** e **ponto como separador de milhar**.  
- Portanto, o pandas não as reconhece inicialmente como numéricas, tratando-as como `object`.  
- Essa constatação é importante: mostra que será necessário realizar **ajustes e conversões de tipos de dados** na etapa de **Data Preparation**.  

➡️ Aqui, entretanto, nosso objetivo é apenas **entender o dataset original**:  
vemos que os anos variam de **2006 a 2020** e que a quantidade de contribuintes por centil está corretamente registrada,  
confirmando a granularidade da base.  
